# Original course experiment
Ryan Balech and Gabriel Dreik. Archived source, with local paths made portable and outputs cleared.
Recorded aggregate results are in `results/original_notebook.json`. The maintained pipeline is under `src/churn/`.
Requires the original course Parquet files; run from this notebook directory or set `CHURN_DATA_DIR`.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score

# ============================================================
# 0. Paths (change if you need)
# ============================================================
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("CHURN_DATA_DIR", "../data"))
train_path = DATA_DIR / "train.parquet"
test_path = DATA_DIR / "test.parquet"
example_path = DATA_DIR / "example_submission.csv"
output_path = DATA_DIR / "submission_FINAL_42perc_GB_RF.csv"

POS_RATE = 0.42  # ≈ 42% of users predicted as 1 (best on Kaggle)


# ============================================================
# 1. Load data
# ============================================================
train = pd.read_parquet(train_path)
test = pd.read_parquet(test_path)

print("Train shape:", train.shape)
print("Test shape :", test.shape)

# basic cleaning
train = train[train["userId"].notnull()].copy()
test = test[test["userId"].notnull()].copy()
train["userId"] = train["userId"].astype(int)
test["userId"] = test["userId"].astype(int)

# timestamps
for df in [train, test]:
    df["ts"] = pd.to_datetime(df["ts"], unit="ms")
    df["registration"] = pd.to_datetime(df["registration"])
    df["time"] = pd.to_datetime(df["time"])

print("Train time range:", train["time"].min(), "→", train["time"].max())
print("Test  time range:", test["time"].min(), "→", test["time"].max())


# ============================================================
# 2. Labels: will the user cancel in the 10 days after cutoff?
# ============================================================
max_time_train = train["time"].max()
cutoff = max_time_train - pd.Timedelta(days=10)

print("Max time in train:", max_time_train)
print("Cutoff (start of last 10 days):", cutoff)

# observation window (for features)
train_obs = train[train["time"] <= cutoff].copy()
# future window (for labels)
train_future = train[(train["time"] > cutoff) & (train["time"] <= max_time_train)].copy()

# users who cancel in future 10 days
future_cancel_users = (
    train_future.loc[train_future["page"] == "Cancellation Confirmation", "userId"]
    .dropna()
    .astype(int)
    .unique()
)

# users we have in obs window
obs_users = train_obs["userId"].dropna().astype(int).unique()

labels_window = pd.DataFrame({"id": obs_users})
labels_window["target"] = labels_window["id"].isin(future_cancel_users).astype(int)

print("Label distribution:")
print(labels_window["target"].value_counts(normalize=True))


# ============================================================
# 3. Advanced feature engineering (v7)
# ============================================================
def build_features_v7(df, observation_end):
    df = df.copy()

    # helper columns
    df["date"] = df["ts"].dt.date
    df["hour"] = df["ts"].dt.hour
    df["weekday"] = df["ts"].dt.weekday
    df["is_song"] = (df["page"] == "NextSong").astype(int)
    df["is_error"] = (df["status"] != 200).astype(int)

    g = df.groupby("userId")
    feat = pd.DataFrame()

    # ---- core usage ----
    feat["n_events"] = g.size()
    feat["n_songs"] = g["is_song"].sum()
    feat["n_unique_songs"] = g["song"].nunique()
    feat["n_unique_artists"] = g["artist"].nunique()
    feat["n_sessions"] = g["sessionId"].nunique()
    feat["active_days"] = g["date"].nunique()
    feat["total_length"] = g["length"].sum()
    feat["avg_length"] = g["length"].mean()

    feat["events_per_day"] = feat["n_events"] / feat["active_days"].replace(0, 1)
    feat["songs_per_day"] = feat["n_songs"] / feat["active_days"].replace(0, 1)
    feat["sessions_per_day"] = feat["n_sessions"] / feat["active_days"].replace(0, 1)

    # ---- errors ----
    feat["n_errors"] = g["is_error"].sum()
    feat["errors_per_event"] = feat["n_errors"] / feat["n_events"].replace(0, 1)
    feat["errors_per_session"] = feat["n_errors"] / feat["n_sessions"].replace(0, 1)

    # ---- profile ----
    feat["gender_M"] = g["gender"].first().eq("M").astype(int)

    first_reg = g["registration"].first()
    feat["days_since_reg"] = (observation_end - first_reg).dt.days

    first_ts = g["ts"].min()
    last_ts = g["ts"].max()
    feat["activity_span_days"] = (last_ts - first_ts).dt.days
    feat["time_since_last_event"] = (observation_end - last_ts).dt.days

    feat["paid_ratio"] = g["level"].apply(lambda x: (x == "paid").mean())
    feat["last_level_paid"] = g["level"].last().eq("paid").astype(int)
    feat["level_changes"] = (g["level"].nunique() > 1).astype(int)

    # ---- page counts (no Cancellation Confirmation) ----
    pages_to_count = [
        "NextSong",
        "Thumbs Up",
        "Thumbs Down",
        "Roll Advert",
        "Add Friend",
        "Add to Playlist",
        "Error",
        "Help",
        "Settings",
        "Upgrade",
        "Downgrade",
        "Logout",
    ]
    df_pages = df[df["page"].isin(pages_to_count)]

    page_counts = df_pages.pivot_table(
        index="userId", columns="page", values="ts", aggfunc="count"
    ).fillna(0)
    page_counts.columns = [f"page_{c.replace(' ', '_')}" for c in page_counts.columns]
    feat = feat.join(page_counts, how="left").fillna(0)

    up = feat.get("page_Thumbs_Up", pd.Series(0, index=feat.index))
    down = feat.get("page_Thumbs_Down", pd.Series(0, index=feat.index))
    feat["thumbs_up_ratio"] = up / (up + down + 1)

    help_c = feat.get("page_Help", pd.Series(0, index=feat.index))
    advert_c = feat.get("page_Roll_Advert", pd.Series(0, index=feat.index))
    feat["help_rate"] = help_c / feat["n_events"].replace(0, 1)
    feat["advert_rate"] = advert_c / feat["n_events"].replace(0, 1)

    upg = feat.get("page_Upgrade", pd.Series(0, index=feat.index))
    dng = feat.get("page_Downgrade", pd.Series(0, index=feat.index))
    feat["upgrade_count"] = upg
    feat["downgrade_count"] = dng
    feat["net_upgrade"] = upg - dng

    # ---- session statistics ----
    session_sizes = df.groupby(["userId", "sessionId"]).size()
    sess_agg = session_sizes.groupby("userId").agg(["mean", "max", "min"])
    sess_agg.columns = ["mean_sess_len", "max_sess_len", "min_sess_len"]
    feat = feat.join(sess_agg, how="left")

    sess_error = (
        df[df["is_error"] == 1].groupby(["userId", "sessionId"]).size().groupby("userId").size()
    )
    feat["sessions_with_error"] = sess_error
    feat["sessions_with_error_ratio"] = feat["sessions_with_error"].fillna(0) / feat[
        "n_sessions"
    ].replace(0, 1)

    # ---- time of week ----
    weekend_events = df[df["weekday"] >= 5].groupby("userId").size()
    feat["weekend_event_ratio"] = weekend_events / feat["n_events"].replace(0, 1)

    night_events = df[df["hour"] < 6].groupby("userId").size()
    feat["night_event_ratio"] = night_events / feat["n_events"].replace(0, 1)

    # ---- recency: last 3 days ----
    recent_cut = observation_end - pd.Timedelta(days=3)
    df_recent = df[df["ts"] >= recent_cut]
    g_recent = df_recent.groupby("userId")

    recent = pd.DataFrame()
    recent["n_events_3d"] = g_recent.size()
    recent["n_songs_3d"] = g_recent["is_song"].sum()
    recent["active_days_3d"] = g_recent["date"].nunique()
    recent["n_sessions_3d"] = g_recent["sessionId"].nunique()

    feat = feat.join(recent, how="left").fillna(0)

    feat["events_3d_ratio"] = feat["n_events_3d"] / feat["n_events"].replace(0, 1)
    feat["songs_3d_ratio"] = feat["n_songs_3d"] / feat["n_songs"].replace(0, 1)
    feat["sessions_3d_ratio"] = feat["n_sessions_3d"] / feat["n_sessions"].replace(0, 1)

    # ---- last page flags ----
    last_page = g["page"].last()
    feat["last_is_song"] = last_page.eq("NextSong").astype(int)
    feat["last_is_help"] = last_page.eq("Help").astype(int)
    feat["last_is_upgrade"] = last_page.eq("Upgrade").astype(int)
    feat["last_is_downgrade"] = last_page.eq("Downgrade").astype(int)
    feat["last_is_logout"] = last_page.eq("Logout").astype(int)

    # final cleanup
    feat = feat.fillna(0)
    feat.reset_index(inplace=True)
    feat.rename(columns={"userId": "id"}, inplace=True)

    return feat


# ============================================================
# 4. Build features for train (obs window) and test
# ============================================================
train_features_v7 = build_features_v7(train_obs, observation_end=cutoff)
obs_end_test = test["time"].max()
test_features_v7 = build_features_v7(test, observation_end=obs_end_test)

print("train_features_v7:", train_features_v7.shape)
print("test_features_v7 :", test_features_v7.shape)


# ============================================================
# 5. Train / validate GB + RF ensemble
# ============================================================
data_v7 = train_features_v7.merge(labels_window, on="id", how="inner")
print("data_v7 shape:", data_v7.shape)

X = data_v7.drop(columns=["id", "target"])
y = data_v7["target"].values

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

gb = GradientBoostingClassifier(
    n_estimators=600, learning_rate=0.04, max_depth=3, subsample=0.9, random_state=42
)
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=10,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)

gb.fit(X_train, y_train)
rf.fit(X_train, y_train)

val_proba_gb = gb.predict_proba(X_valid)[:, 1]
val_proba_rf = rf.predict_proba(X_valid)[:, 1]
val_proba_ens = 0.7 * val_proba_gb + 0.3 * val_proba_rf

print("Validation AUC (GB): ", roc_auc_score(y_valid, val_proba_gb))
print("Validation AUC (RF): ", roc_auc_score(y_valid, val_proba_rf))
print("Validation AUC (ENS):", roc_auc_score(y_valid, val_proba_ens))


# ============================================================
# 6. Train on full data and predict on test
# ============================================================
X_full = X
y_full = y
X_test = test_features_v7.drop(columns=["id"])

gb_full = GradientBoostingClassifier(
    n_estimators=600, learning_rate=0.04, max_depth=3, subsample=0.9, random_state=42
)
rf_full = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=10,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)

gb_full.fit(X_full, y_full)
rf_full.fit(X_full, y_full)

test_proba_gb = gb_full.predict_proba(X_test)[:, 1]
test_proba_rf = rf_full.predict_proba(X_test)[:, 1]
test_proba_ens = 0.7 * test_proba_gb + 0.3 * test_proba_rf

pred = pd.DataFrame({"id": test_features_v7["id"], "proba_ens": test_proba_ens})


# ============================================================
# 7. Apply 42% rule and save final submission
# ============================================================
example = pd.read_csv(example_path)
merged = example[["id"]].merge(pred, on="id", how="left")

print("Missing probs after merge:", merged["proba_ens"].isna().sum())
merged["proba_ens"] = merged["proba_ens"].fillna(0)

print(merged["proba_ens"].describe())

# choose threshold so that about POS_RATE users are predicted as 1
threshold = np.quantile(merged["proba_ens"], 1 - POS_RATE)
print(f"Chosen POS_RATE = {POS_RATE:.2f}, threshold = {threshold:.6f}")

merged["target"] = (merged["proba_ens"] >= threshold).astype(int)
print("Final class distribution:")
print(merged["target"].value_counts(normalize=True))

submission = merged[["id", "target"]]
submission.to_csv(output_path, index=False)

print("Saved final submission to:", output_path)
print(submission.head())

In [ ]:
import numpy as np
import pandas as pd

# choose new percentage
p = 0.424  # 41.8%

# recompute threshold
thr = np.quantile(merged["proba_ens"], 1 - p)
print("New threshold:", thr)

# new binary target
merged["target"] = (merged["proba_ens"] >= thr).astype(int)
print("New class distribution:")
print(merged["target"].value_counts(normalize=True))

# save new submission
out_path = DATA_DIR / "submission_0.424_GB_RF.csv"
submission = merged[["id", "target"]]
submission.to_csv(out_path, index=False)
print("Saved:", out_path)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, accuracy_score

sns.set(style="whitegrid")

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x="target", data=data_v7)
plt.title("Label distribution (0 = non-churn, 1 = churn)")
plt.xlabel("Target")
plt.ylabel("Number of users")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=data_v7, x="target", color="steelblue")
plt.title("Label Distribution")
plt.xlabel("Churn Label (0 = No, 1 = Yes)")
plt.ylabel("Number of Users")
plt.tight_layout()
plt.show()

In [ ]:
imp_df = (
    pd.DataFrame({"feature": X_full.columns, "importance": rf_full.feature_importances_})
    .sort_values("importance", ascending=False)
    .head(15)
)

plt.figure(figsize=(10, 6))
sns.barplot(data=imp_df, x="importance", y="feature", color="teal")
plt.title("Top 15 Feature Importances (Random Forest)")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("feature_imp.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
fpr_gb, tpr_gb, _ = roc_curve(y_valid, val_proba_gb)
fpr_rf, tpr_rf, _ = roc_curve(y_valid, val_proba_rf)
fpr_en, tpr_en, _ = roc_curve(y_valid, val_proba_ens)

plt.figure(figsize=(7, 7))

sns.lineplot(x=fpr_gb, y=tpr_gb, label=f"GB (AUC={auc(fpr_gb, tpr_gb):.3f})")
sns.lineplot(x=fpr_rf, y=tpr_rf, label=f"RF (AUC={auc(fpr_rf, tpr_rf):.3f})")
sns.lineplot(x=fpr_en, y=tpr_en, label=f"Ensemble (AUC={auc(fpr_en, tpr_en):.3f})")

plt.plot([0, 1], [0, 1], "k--", alpha=0.6)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves on Validation Set")
plt.legend()
plt.tight_layout()

# ✅ SAVE FIRST
plt.savefig("ROC_curves.png", dpi=300, bbox_inches="tight")

# Optional: show on screen
plt.show()

plt.close()

In [ ]:
ps = np.linspace(0.30, 0.50, 21)
accs = []

for p in ps:
    thr = np.quantile(val_proba_ens, 1 - p)
    preds = (val_proba_ens >= thr).astype(int)
    accs.append(accuracy_score(y_valid, preds))

plt.figure(figsize=(8, 5))
sns.lineplot(x=ps, y=accs, marker="o", color="steelblue")
plt.axvline(0.424, color="red", linestyle="--", label="Chosen p = 0.424")
plt.title("Accuracy vs Positive Rate (Quantile Thresholding)")
plt.xlabel("Positive Rate p (Predicted % Churners)")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
# ✅ SAVE FIRST
plt.savefig("AccvsPR.png", dpi=300, bbox_inches="tight")

# Optional: show on screen
plt.show()

plt.close()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(merged["proba_ens"], bins=40, kde=True, color="steelblue")
plt.title("Distribution of Ensemble Churn Probabilities")
plt.xlabel("Predicted Probability")
plt.ylabel("Number of Users")
plt.tight_layout()
plt.show()

In [ ]:
p = 0.424
thr = np.quantile(merged["proba_ens"], 1 - p)

plt.figure(figsize=(8, 5))
sns.histplot(merged["proba_ens"], bins=40, kde=False, color="teal")
plt.axvline(thr, color="red", linestyle="--", linewidth=2, label=f"Threshold = {thr:.4f}")
plt.title("Ensemble Probability Distribution with Final Threshold")
plt.xlabel("Predicted Probability")
plt.ylabel("User Count")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set(style="whitegrid", context="talk")

start_time = train["time"].min()
end_time = train["time"].max()

fig, ax = plt.subplots(figsize=(12, 3.2), constrained_layout=True)

# main timeline
ax.hlines(y=0, xmin=start_time, xmax=end_time, color="black", linewidth=2)

# observation window shaded
ax.axvspan(
    start_time, cutoff, ymin=0.4, ymax=0.6, color="skyblue", alpha=0.5, label="Observation Window"
)

# future window shaded
ax.axvspan(cutoff, end_time, ymin=0.4, ymax=0.6, color="salmon", alpha=0.5, label="Future Window")

# cutoff line
ax.axvline(cutoff, color="red", linestyle="--", linewidth=2)

# x-axis date formatting (reduce ticks to avoid crowding)
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

# no y-axis
ax.set_yticks([])

ax.set_title("Timeline of Observation and Future Windows")
ax.set_xlabel("Date")

ax.set_ylim(-0.2, 1.0)
ax.legend(loc="upper left", frameon=True)

plt.show()